In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.6 MB/s eta 0:00:00


In [ ]:
import torch
print("¿GPU Disponible?:", torch.cuda.is_available())
print("Nombre de la GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Ninguna")

¿GPU Disponible?: True
Nombre de la GPU: Tesla T4


In [ ]:
import os
from pathlib import Path

# Descomprimir el archivo ZIP
!unzip -q /content/drive/MyDrive/Residencia/DatasetResidencia-2808.v1-version1-definitivaresidencia.yolov11.zip -d /content/mi_dataset

ruta_local = Path('/content/mi_dataset')

# Si al comprimir se creó una carpeta interna tipo 'Dataset3', la buscamos automáticamente
contenidos = os.listdir(ruta_local)
if len(contenidos) == 1 and (ruta_local / contenidos[0]).is_dir():
    ruta_local = ruta_local / contenidos[0]

# Contar de forma recursiva TODOS los archivos de imagen (jpg, jpeg, png) para verificar las 11,000
extensiones_imagen = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
total_imagenes = sum(len(list(ruta_local.rglob(ext))) for ext in extensiones_imagen)

print(f"¡Proceso completado con éxito!")
print(f"Ruta raíz efectiva del dataset: {ruta_local}")
print(f"Se han encontrado {total_imagenes} imágenes en total dentro del almacenamiento de Colab.")

replace /content/mi_dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/mi_dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
¡Proceso completado con éxito!
Ruta raíz efectiva del dataset: /content/mi_dataset
Se han encontrado 28139 imágenes en total dentro del almacenamiento de Colab.


In [ ]:
import yaml

ruta_yaml = '/content/mi_dataset/data.yaml'

# 1. Leer el archivo yaml actual
with open(ruta_yaml, 'r') as f:
    datos_yaml = yaml.safe_load(f)

# 2. Sobrescribir las rutas con la ubicación real dentro de Colab
datos_yaml['path'] = '/content/mi_dataset' # Ruta raíz del dataset
datos_yaml['train'] = 'train/images' # Subcarpeta de entrenamiento
datos_yaml['val'] = 'valid/images'   # Subcarpeta de validación
datos_yaml['test'] = 'test/images'   # Subcarpeta de prueba (si existe)

# 3. Guardar los cambios en el archivo
with open(ruta_yaml, 'w') as f:
    yaml.dump(datos_yaml, f, default_flow_style=False)

print("¡Archivo data.yaml corregido con éxito para Google Colab!")

¡Archivo data.yaml corregido con éxito para Google Colab!


In [ ]:

import sys
import os

In [ ]:
# 1. Desinstalar por completo la librería 'ultralytics' global que genera el conflicto
!pip uninstall -y ultralytics

# 2. Nos posicionamos en la segunda carpeta (donde está tu código de LeYOLO completo)
#%cd /content/drive/.shortcut-targets-by-id/1eDVeBz5nk32HGbR7tvjyd6B3cd7uMYzv/Residencia/LeYOLO
%cd /content/drive/.shortcut-targets-by-id/11tCgdY44nG10Pu_nJucRkeoP2ciIS1j4/Residencia/LeYOLO
# 3. Forzar a Python a buscar todos los módulos (incluyendo .conv) aquí dentro
import sys
import os
sys.path.insert(0, os.getcwd())

print("¡Entorno limpio de librerías globales en conflicto!")

Found existing installation: ultralytics 8.4.140
Uninstalling ultralytics-8.4.140:
  Successfully uninstalled ultralytics-8.4.140
/content/drive/.shortcut-targets-by-id/11tCgdY44nG10Pu_nJucRkeoP2ciIS1j4/Residencia/LeYOLO
¡Entorno limpio de librerías globales en conflicto!


In [ ]:
import sys
import os

# 1. DESACTIVAR WANDB A NIVEL DE SISTEMA OPERATIVO
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

import torch

# 2. PARCHE MAESTRO PARA PYTORCH 2.6+ (Corregido con protección anti-recursión)
if not hasattr(torch, '_patched_for_weights'):
    _old_torch_load = torch.load
    def _patched_torch_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _old_torch_load(*args, **kwargs)
    torch.load = _patched_torch_load
    torch._patched_for_weights = True
    print("¡Parche de compatibilidad de PyTorch aplicado con éxito!")

# 3. IMPORTAR LIBRERÍA LOCAL Y SETTINGS
from ultralytics import YOLO, settings

# Desactivar wandb en Ultralytics
settings.update({'wandb': False})
print("¡LeYOLO importado correctamente!")

# 4. CARGAR ARQUITECTURA DEL MODELO PERSONALIZADO
ruta_arquitectura_custom = "ultralytics/cfg/cfg/leyolomedium_custom.yaml"
model = YOLO(ruta_arquitectura_custom)
print("¡Arquitectura personalizada cargada con éxito!")

# Limpiar callbacks residuales de wandb
if hasattr(model, 'callbacks'):
    for key in list(model.callbacks.keys()):
        model.callbacks[key] = [cb for cb in model.callbacks[key] if 'wandb' not in getattr(cb, '__module__', '')]

# 5. INICIAR ENTRENAMIENTO (Primeras 10 épocas)
print("Iniciando entrenamiento en GPU...")
results = model.train(
    data='/content/mi_dataset/data.yaml',
    pretrained='weights/LeYOLOMedium.pt',
    epochs=10, # <--- Solo 10 épocas
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    patience=20,
    project="LeYOLO_Residencia",
    name="LeYOLO_Medium_Custom"
)

# 6. COPIA DIRECTA A TU CARPETA EXISTENTE 'Residencia'
!cp -r LeYOLO_Residencia /content/drive/MyDrive/Residencia/
print("¡Primeras 10 épocas guardadas con éxito en Google Drive!")

¡Parche de compatibilidad de PyTorch aplicado con éxito!
¡LeYOLO importado correctamente!
WARNING ⚠️ no model scale passed. Assuming scale='m'.
¡Arquitectura personalizada cargada con éxito!
Iniciando entrenamiento en GPU...
New https://pypi.org/project/ultralytics/8.4.140 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.1.4 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=ultralytics/cfg/cfg/leyolomedium_custom.yaml, data=/content/mi_dataset/data.yaml, epochs=10, time=None, patience=20, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=LeYOLO_Residencia, name=LeYOLO_Medium_Custom3, exist_ok=False, pretrained=weights/LeYOLOMedium.pt, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=False, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio

100%|██████████| 755k/755k [00:00<00:00, 23.8MB/s]


Overriding model.yaml nc=80 with nc=36
WARNING ⚠️ no model scale passed. Assuming scale='m'.

                   from  n    params  module                                       arguments                     
  0                  -1  1       696  ultralytics.nn.modules.conv.Conv             [3, 24, 3, 2]                 
  1                  -1  1      1800  ultralytics.nn.modules.block.InvertedBottleneck[24, 32, 3, 24, None, 'SI', 2]
  2                  -1  1       512  ultralytics.nn.modules.block.SEBlock         [32]                          
  3                  -1  1      9440  ultralytics.nn.modules.block.InvertedBottleneck[32, 96, 3, 64, None, 'SI', 2]
  4                  -1  1      4608  ultralytics.nn.modules.block.SEBlock         [96]                          
  5                  -1  1     58752  ultralytics.nn.modules.block.InvertedBottleneck[96, 192, 3, 192, None, 'SI', 2]
  6                  -1  1     18432  ultralytics.nn.modules.block.SEBlock         [192]            

/content/drive/.shortcut-targets-by-id/11tCgdY44nG10Pu_nJucRkeoP2ciIS1j4/Residencia/LeYOLO/ultralytics/engine/trainer.py:272: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
train: Scanning /content/mi_dataset/train/labels... 22334 images, 65 backgrounds, 54 corrupt: 100%|██████████| 22334/22334 [00:37<00:00, 599.08it/s]

train: WARNING ⚠️ /content/mi_dataset/train/images/121_png.rf.5e652aa3ea0ab67f8ff5cbe881e5a5ed.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /content/mi_dataset/train/images/299_png.rf.da7e5b8db425327c4af99b781b0d12dd.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /content/mi_dataset/train/images/536_png.rf.792afbb106b99ec0f3c09debe5c8f85c.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /content/mi_dataset/train/images/621_png.rf.9bfa1dd53e764dd0aa98363b02ed0ce7.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /content/mi_dataset/train/images/corn-001_jpg.rf.57e28f0dfc71b4d3628603b9de1926e0.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.0093129 1.0015787]
train: WARNING ⚠️ /content/mi_dataset/train/images/corn-004_jpg.rf.0662f5a1dc50638047fdaa5b1a9523f4.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.001048 1.001048 1.008024]
train: WARNING ⚠️ /content/mi_dataset/train/images/corn-005_jpg.rf.c69b31f0f0513b81

...

                   all       3653      34596      0.387      0.137     0.0905      0.041

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      10/10      4.82G      2.122      2.671      1.838         35        640: 100%|██████████| 2785/2785 [11:40<00:00,  3.98it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 229/229 [01:12<00:00,  3.14it/s]


                   all       3653      34596      0.386      0.143     0.0939     0.0431

10 epochs completed in 2.235 hours.
Optimizer stripped from LeYOLO_Residencia/LeYOLO_Medium_Custom3/weights/last.pt, 2.0MB
Optimizer stripped from LeYOLO_Residencia/LeYOLO_Medium_Custom3/weights/best.pt, 2.0MB

Validating LeYOLO_Residencia/LeYOLO_Medium_Custom3/weights/best.pt...
Ultralytics YOLOv8.1.4 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
installed
FLOP(G):  3.7784064
leYOLOmedium_custom summary: 165 layers, 940685 parameters, 0 gradients, 3.778406 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 229/229 [01:14<00:00,  3.06it/s]


                   all       3653      34596      0.386      0.143      0.094      0.043
                 Apple       3653       7616      0.314      0.535      0.397      0.207
               Apricot       3653        559          0          0    0.00741    0.00418
             Aubergine       3653        144      0.105      0.701      0.333      0.129
                Banana       3653         62      0.496     0.0484     0.0903     0.0478
                  Bean       3653        436          1          0     0.0139    0.00336
                Carrot       3653        615      0.156       0.53      0.241     0.0822
                Cherry       3653       3883     0.0259   0.000258    0.00349    0.00204
             Courgette       3653         61          1          0          0          0
           DragonFruit       3653        259      0.117     0.0327     0.0401     0.0138
                Durian       3653       1668      0.323      0.173      0.151     0.0605
                   Fi

CUANDO SE TERMINA LA SESIO DE GOOGLE COLAB Y SE NECESITA VOLVER A ENTRENAR EJECUTAR A PARTIR DE ESTA **CELDA**

In [1]:
import sys
import os

# 1. MONTAR GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2. MOVERSE AL DIRECTORIO DE TRABAJO (Ajusta la ruta si difiere)
%cd /content/drive/.shortcut-targets-by-id/11tCgdY44nG10Pu_nJucRkeoP2ciIS1j4/Residencia/LeYOLO

# 3. DESACTIVAR WANDB
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

import torch

# 4. PARCHE DE PYTORCH (Anti-recursión)
if not hasattr(torch, '_patched_for_weights'):
    _old_torch_load = torch.load
    def _patched_torch_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _old_torch_load(*args, **kwargs)
    torch.load = _patched_torch_load
    torch._patched_for_weights = True
    print("¡Parche de PyTorch aplicado con éxito!")

from ultralytics import YOLO, settings
settings.update({'wandb': False})
print("¡Entorno configurado y listo!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/11tCgdY44nG10Pu_nJucRkeoP2ciIS1j4/Residencia/LeYOLO
¡Parche de PyTorch aplicado con éxito!
¡Entorno configurado y listo!


In [2]:
import os
from pathlib import Path

# Descomprimir el archivo ZIP
!unzip -q /content/drive/MyDrive/Residencia/DatasetResidencia-2808.v1-version1-definitivaresidencia.yolov11.zip -d /content/mi_dataset

ruta_local = Path('/content/mi_dataset')

# Si al comprimir se creó una carpeta interna tipo 'Dataset3', la buscamos automáticamente
contenidos = os.listdir(ruta_local)
if len(contenidos) == 1 and (ruta_local / contenidos[0]).is_dir():
    ruta_local = ruta_local / contenidos[0]

# Contar de forma recursiva TODOS los archivos de imagen (jpg, jpeg, png) para verificar las 11,000
extensiones_imagen = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
total_imagenes = sum(len(list(ruta_local.rglob(ext))) for ext in extensiones_imagen)

print(f"¡Proceso completado con éxito!")
print(f"Ruta raíz efectiva del dataset: {ruta_local}")
print(f"Se han encontrado {total_imagenes} imágenes en total dentro del almacenamiento de Colab.")

replace /content/mi_dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/mi_dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
¡Proceso completado con éxito!
Ruta raíz efectiva del dataset: /content/mi_dataset
Se han encontrado 28139 imágenes en total dentro del almacenamiento de Colab.


In [3]:
import yaml

ruta_yaml = '/content/mi_dataset/data.yaml'

# 1. Leer el archivo yaml actual
with open(ruta_yaml, 'r') as f:
    datos_yaml = yaml.safe_load(f)

# 2. Sobrescribir las rutas con la ubicación real dentro de Colab
datos_yaml['path'] = '/content/mi_dataset' # Ruta raíz del dataset
datos_yaml['train'] = 'train/images' # Subcarpeta de entrenamiento
datos_yaml['val'] = 'valid/images'   # Subcarpeta de validación
datos_yaml['test'] = 'test/images'   # Subcarpeta de prueba (si existe)

# 3. Guardar los cambios en el archivo
with open(ruta_yaml, 'w') as f:
    yaml.dump(datos_yaml, f, default_flow_style=False)

print("¡Archivo data.yaml corregido con éxito para Google Colab!")

¡Archivo data.yaml corregido con éxito para Google Colab!


In [ ]:
from ultralytics import YOLO

# 1. Cargar el último punto de control guardado en Google Drive
ruta_last = "/content/drive/MyDrive/Residencia/LeYOLO_Residencia/LeYOLO_Medium_Custom10/weights/last.pt"
model = YOLO(ruta_last)

# 2. Reanudar PASANDO EL MODELO DIRECTAMENTE (Sin resume=True)
# Se especifica 'epochs=20' para extender el límite hasta la época 20
results = model.train(
    data='/content/mi_dataset/data.yaml',
    epochs=20,                      # Amplía el objetivo total a 20 épocas
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    patience=10,
    project="LeYOLO_Residencia",
    name="LeYOLO_Medium_Custom"
)

# 3. Guardar el progreso en Google Drive
!cp -r LeYOLO_Residencia /content/drive/MyDrive/Residencia/
print("¡Bloque de épocas 11-20 completado y respaldado en Google Drive!")

New https://pypi.org/project/ultralytics/8.4.143 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.1.4 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=/content/drive/MyDrive/Residencia/LeYOLO_Residencia/LeYOLO_Medium_Custom10/weights/last.pt, data=/content/mi_dataset/data.yaml, epochs=20, time=None, patience=10, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=LeYOLO_Residencia, name=LeYOLO_Medium_Custom12, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=False, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visuali

100%|██████████| 755k/755k [00:00<00:00, 17.7MB/s]


WARNING ⚠️ no model scale passed. Assuming scale='m'.

                   from  n    params  module                                       arguments                     
  0                  -1  1       696  ultralytics.nn.modules.conv.Conv             [3, 24, 3, 2]                 
  1                  -1  1      1800  ultralytics.nn.modules.block.InvertedBottleneck[24, 32, 3, 24, None, 'SI', 2]
  2                  -1  1       512  ultralytics.nn.modules.block.SEBlock         [32]                          
  3                  -1  1      9440  ultralytics.nn.modules.block.InvertedBottleneck[32, 96, 3, 64, None, 'SI', 2]
  4                  -1  1      4608  ultralytics.nn.modules.block.SEBlock         [96]                          
  5                  -1  1     58752  ultralytics.nn.modules.block.InvertedBottleneck[96, 192, 3, 192, None, 'SI', 2]
  6                  -1  1     18432  ultralytics.nn.modules.block.SEBlock         [192]                         
  7                  -1  

/content/drive/.shortcut-targets-by-id/11tCgdY44nG10Pu_nJucRkeoP2ciIS1j4/Residencia/LeYOLO/ultralytics/engine/trainer.py:272: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
train: Scanning /content/mi_dataset/train/labels... 22334 images, 65 backgrounds, 54 corrupt: 100%|██████████| 22334/22334 [00:39<00:00, 558.65it/s]

train: WARNING ⚠️ /content/mi_dataset/train/images/121_png.rf.5e652aa3ea0ab67f8ff5cbe881e5a5ed.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /content/mi_dataset/train/images/299_png.rf.da7e5b8db425327c4af99b781b0d12dd.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /content/mi_dataset/train/images/536_png.rf.792afbb106b99ec0f3c09debe5c8f85c.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /content/mi_dataset/train/images/621_png.rf.9bfa1dd53e764dd0aa98363b02ed0ce7.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /content/mi_dataset/train/images/corn-001_jpg.rf.57e28f0dfc71b4d3628603b9de1926e0.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.0093129 1.0015787]
train: WARNING ⚠️ /content/mi_dataset/train/images/corn-004_jpg.rf.0662f5a1dc50638047fdaa5b1a9523f4.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.001048 1.001048 1.008024]
train: WARNING ⚠️ /content/mi_dataset/train/images/corn-005_jpg.rf.c69b31f0f0513b81

...

       3/20      5.19G        1.7      1.733      1.328         71        640: 100%|██████████| 2785/2785 [12:23<00:00,  3.75it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 229/229 [01:18<00:00,  2.92it/s]


                   all       3653      34596      0.437      0.356      0.322      0.163

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       4/20      3.79G      1.692      1.729      1.325        152        640: 100%|██████████| 2785/2785 [12:20<00:00,  3.76it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 229/229 [01:18<00:00,  2.91it/s]


                   all       3653      34596      0.443      0.355      0.317      0.158

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       5/20      3.55G      1.693      1.728      1.329         93        640:  20%|██        | 562/2785 [02:30<08:40,  4.27it/s]

In [ ]:
import sys
import os
import torch

from ultralytics import YOLO


# 1. PARCHE SEGURO PARA PYTORCH 2.6+: Evita el bucle infinito de recursión
if not hasattr(torch, '_is_patched'):
    _old_torch_load = torch.load
    def _patched_torch_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _old_torch_load(*args, **kwargs)
    torch.load = _patched_torch_load
    torch._is_patched = True
    print("¡Parche de compatibilidad de PyTorch aplicado de forma segura!")
else:
    print("El parche ya estaba aplicado de forma segura. Saltando para evitar bucle.")

# 2. Asegurar la ruta local correcta de tu LeYOLO interno
%cd /content/drive/.shortcut-targets-by-id/1eDVeBz5nk32HGbR7tvjyd6B3cd7uMYzv/checkpoints_comparativa/LeYOLO_Medium_Custom6
sys.path.insert(0, os.getcwd())

from ultralytics import YOLO

# 3. Cargar el MEJOR modelo guardado del entrenamiento anterior (best.pt)
path_best_weights = "/weights/best.pt"
model = YOLO(path_best_weights)

print("Iniciando validación del modelo entrenado...")

# 4. Ejecutar el modo de validación básica
metrics = model.val(
    data='/content/mi_dataset/datasetBalanceado-v3-2/data.yaml', # Tu archivo YAML
    imgsz=640,
    batch=16,
    device=0,                                                    # Usar GPU
    split='val'                                                  # Evaluar sobre el conjunto de validación
)

print("\n--- RESULTADOS DE LA VALIDACIÓN ---")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precisión (Precision): {metrics.box.mp:.4f}")
print(f"Sensibilidad (Recall): {metrics.box.mr:.4f}")

El parche ya estaba aplicado de forma segura. Saltando para evitar bucle.
[Errno 2] No such file or directory: '/content/drive/.shortcut-targets-by-id/1eDVeBz5nk32HGbR7tvjyd6B3cd7uMYzv/checkpoints_comparativa/LeYOLO_Medium_Custom6'
/content/drive/.shortcut-targets-by-id/1eDVeBz5nk32HGbR7tvjyd6B3cd7uMYzv/Datasets-Memoria/LeYOLO


FileNotFoundError: [Errno 2] No such file or directory: '/weights/best.pt'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
import os
import torch
import time

# 1. Mantener el parche seguro de PyTorch 2.6+
if not hasattr(torch, '_is_patched'):
    _old_torch_load = torch.load
    def _patched_torch_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _old_torch_load(*args, **kwargs)
    torch.load = _patched_torch_load
    torch._is_patched = True

# 2. Configurar rutas de LeYOLO
ruta_base = "/content/drive/.shortcut-targets-by-id/1eDVeBz5nk32HGbR7tvjyd6B3cd7uMYzv/Datasets-Memoria/PruebaDatasetv3-2/LeYOLO/LeYOLO"
%cd {ruta_base}
sys.path.insert(0, os.getcwd())

from ultralytics import YOLO

# 3. Cargar el mejor modelo que entrenaste
path_best_weights = "runs/detect/train/weights/best.pt"
model = YOLO(path_best_weights)

# 4. Origen de la imagen para la prueba
origen_prueba = "https://ultralytics.com/images/bus.jpg"

print("Calentando la GPU (Warmup)...")
# Hacemos una primera pasada para inicializar CUDA a BS=1
model.predict(source=origen_prueba, imgsz=640, device=0, verbose=False)

print("Iniciando Benchmark oficial a Batch Size = 1...")
tiempos_puros_gpu = []
num_iteraciones = 50  # Hacemos 50 pasadas para estabilizar el promedio

for _ in range(num_iteraciones):
    # Forzamos stream=True y procesamos una sola imagen por paso (batch=1)
    resultados = model.predict(source=origen_prueba, imgsz=640, device=0, verbose=False)

    # Extraemos el tiempo de inferencia puro que tomó la GPU en milisegundos
    tiempo_inf = resultados[0].speed['inference']
    tiempos_puros_gpu.append(tiempo_inf)

# 5. Calcular métricas finales
tiempo_promedio_ms = sum(tiempos_puros_gpu) / len(tiempos_puros_gpu)
tiempo_promedio_segundos = tiempo_promedio_ms / 1000.0

# Calcular FPS a BS=1
fps_bs1 = 1 / tiempo_promedio_segundos

print("\n" + "="*45)
print("   BENCHMARK LeYOLO (BATCH SIZE = 1)   ")
print("="*45)
print(f"Hardware utilizado:            Tesla T4 (GPU)")
print(f"Iteraciones evaluadas:         {num_iteraciones}")
print(f"Tiempo de Inferencia promedio:  {tiempo_promedio_ms:.2f} ms por frame")
print(f"RENDIMIENTO EN TIEMPO REAL:    {fps_bs1:.2f} FPS")
print("="*45)

/content/drive/.shortcut-targets-by-id/1eDVeBz5nk32HGbR7tvjyd6B3cd7uMYzv/Datasets-Memoria/PruebaDatasetv3-2/LeYOLO/LeYOLO
Calentando la GPU (Warmup)...
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Iniciando Benchmark oficial a Batch Size = 1...
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
Found https://ultralytics.com/im